# Sendo a base apresentada no arquivo abaixo:
- https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce?select=olist_customers_dataset.csv
    - Já disponível no banco de dados `vendas_db.db` através do link:
        - https://drive.google.com/file/d/1eONzrbEu5BoijDRj56gjL_2Ik5qxtR6U/view?usp=sharing
<br><br>
- Sua tarefa é ajudar a área de negócios a **montar uma apresentação para a diretoria** para provar a **necessidade de investistimento em uma área de melhoria da experência do cliente ao ter um atraso na entrega**
<br><br>
- Algumas considerações são importantes
    - O **time de logística não considera que o atraso na entrega é um problema relevante** e falou que, em média, as entregas estão sendo feitas 10 dias antes do prazo combinado
    - Não é desejado a previsão de uma entrega atrasada, apenas a **exposição que esse é um problema que pode impactar os clientes**
    - Não queremos uma abordagem de: "nenhuma entrega pode atrasar". Vamos ser mais tranquilos e seguir na linha de: **"uma entrega pode atrasar. Como eu posso melhorar a experiência do cliente caso isso aconteça?"**

### Preparando nosso "ambiente"

In [7]:
# Importando o sqlite3
import sqlite3

In [8]:
# Importando o pandas
import pandas as pd

In [9]:
# Criando uma conexão
con = sqlite3.connect('../data/vendas_db.db')

In [10]:
# E então criando o cursor
cur = con.cursor()

In [11]:
# Usando a função python que já havíamos criado
def executa_consulta(consulta):
    resultado = cur.execute(consulta).fetchall()
    resultado = pd.DataFrame(resultado)
    colunas = [i[0] for i in cur.description]
    if resultado.shape[1] > 0:
        resultado.columns = colunas
    print(resultado.shape)
    display(resultado.head(3))
    return resultado

### <font color='blue'> 2. Existe relação da avaliação do cliente com o atraso do pedido? </font>

In [12]:
# Vamos começar visualizando a tabela de avaliações
avaliacoes = executa_consulta('SELECT * FROM order_reviews')

(99224, 8)


,index,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24


In [13]:
# Verificando a quantidade de avaliações em cada uma das notas
avaliacoes.review_score.value_counts().sort_index()

review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64

In [14]:
# Visualizando em percentual
round((avaliacoes.review_score.value_counts().sort_index()/avaliacoes.shape[0])*100,1)

review_score
1    11.5
2     3.2
3     8.2
4    19.3
5    57.8
Name: count, dtype: float64

In [15]:
# Podemos também visualizar a tabela de pedidos
pedidos = executa_consulta('SELECT * FROM orders')

(99441, 9)


,index,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00


In [16]:
# Relacionando as duas tabelas
avaliacao_ordem = executa_consulta('SELECT * FROM orders o \
                                   LEFT JOIN order_reviews ore \
                                   ON o.order_id = ore.order_id')

(99992, 17)


,index,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,index,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,32790.0,a54f0611adc9ed256b57ede6b6eb5114,e481f51cbdc54678b7cc49136f2d6af7,4.0,NaN,"Não testei o produto ainda, mas ele veio corre...",2017-10-11 00:00:00,2017-10-12 03:43:48
1,1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,29158.0,8d5266042046a06655c8db133d120ba5,53cdb2fc8bc7dce0b6741e2150273451,4.0,Muito boa a loja,Muito bom o produto.,2018-08-08 00:00:00,2018-08-08 18:37:50
2,2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,4323.0,e73b67b67587f7644d5bd1a52deb1b01,47770eb9100c2d0c44946d9cf07ec65d,5.0,NaN,NaN,2018-08-18 00:00:00,2018-08-22 19:07:58


In [17]:
# Podemos pegar apenas as colunas que quisermos utilizar

In [18]:
# Todos os pedidos possuem avaliação?

In [19]:
# Verificando a diferença entre as duas tabelas

**Eu tenho 768 pedidos sem avaliação mas apenas 217 linhas a mais**<br>
**Será que algum pedido possui mais de 1 avaliação?**<br>

In [20]:
# Visualizando se alguma ordem apareceu mais de 1 vez

In [21]:
# Verificando essa primeira ordem na tabela de pedidos

In [22]:
# Agora verificando na tabela de avaliações

**Agora precisamos tomar uma decisão de negócio!**<br>
**O que fazer com registros com mais de 1 linha?**

In [23]:
# E se usarmos a média?

In [24]:
# Verificando as notas

**O que podemos fazer com essas médias com valores não inteiros?**

In [25]:
# E se usarmos o máximo?

In [26]:
# Verificando as notas

**Podemos trazer do SQL tanto a média quanto o máximo e então aprofundar na análise**

In [27]:
# Primeiramente fazendo o join entre as tabelas

In [28]:
# Verificando as informações da base

In [29]:
# Transformando as colunas de data em datas

In [30]:
# Verificando novamente as informações da base

In [31]:
# Verificando as primeiras linhas da tabela

In [32]:
# Analisando o impacto de usar o máximo ao invés da média

In [33]:
# Retirando quando maximo é igual a média

In [34]:
# Verificando o % dessas notas

**Podemos definir com o negócio que vamos utilizar o máximo da nota**

In [35]:
# Vamos considerar uma única coluna de nota

In [36]:
# E apagar as outras colunas para evitar confusão

### <font color='blue'>Agora sim vamos responder a pergunta da relação entre o atraso e a avaliação</font>

In [37]:
# Calculando o atraso na entrega

In [38]:
# Marcando entregas que tiveram atraso com uma marcação (flag)

In [39]:
# Verificando novamente a base

In [40]:
# Verificando a nota quando existe atraso e quando não existe atraso

In [41]:
# Analisando os atrasos em relação a média das notas

In [42]:
# Melhorando o gráfico e visualizando a quantidade de registros em cada dia

In [43]:
# Será que podemos agrupar os atrasos?

In [44]:
# Aplicando a função

In [45]:
# Verificando a média das notas por faixa de atraso

In [46]:
# Usando a função já pronta apenas para adiantar a escrita do código
def agrupa_atraso(atraso):
    if atraso < -20:
        return '01. Mais de 20 dias de atraso'
    elif atraso < -15:
        return '02. Entre 15 e 20 dias de atraso'
    elif atraso < -10:
        return '03. Entre 10 e 15 dias de atraso'
    elif atraso < -8:
        return '04. Entre 8 e 10 dias de atraso'
    elif atraso < -6:
        return '05. Entre 6 e 8 dias de atraso'
    elif atraso < -4:
        return '06. Entre 4 e 6 dias de atraso'
    elif atraso < -2:
        return '07. Entre 2 e 4 dias de atraso'
    elif atraso < 0:
        return '08. Entre 1 e 2 dias de atraso'
    elif atraso == 0:
        return '09. Entregue na data'
    elif atraso <= 2:
        return '10. Entre 0 e 2 dias antes do prazo'
    elif atraso <= 4:
        return '11. Entre 2 e 4 dias antes do prazo'
    elif atraso <= 6:
        return '12. Entre 4 e 6 dias antes do prazo' 
    elif atraso <= 8:
        return '13. Entre 6 e 8 dias antes do prazo' 
    elif atraso <= 10:
        return '14. Entre 8 e 10 dias antes do prazo' 
    elif atraso <= 15:
        return '15. Entre 10 e 15 dias antes do prazo' 
    elif atraso <= 20:
        return '16. Entre 15 e 20 dias antes do prazo' 
    elif atraso <= 30:
        return '17. Entre 20 e 30 dias antes do prazo' 
    elif atraso <= 40:
        return '18. Entre 30 e 40 dias antes do prazo' 
    elif atraso > 40:
        return '19. Mais de 40 dias antes do prazo' 
    else:
        return '20. Verificar'

In [47]:
# Aplicando novamente a função

In [48]:
# Visualizando a base

In [49]:
# Verificando as avaliações que não conseguimos analisar a data

In [50]:
# Podemos retirar da base as linhas que não são do nosso interesse

In [51]:
# Analisando a média das notas por faixa de atraso

**Depois que você já visualizou essa informação, pense na melhor maneira de apresentar**

In [52]:
# Salvando em uma variavel

In [53]:
# Visualizando graficamente

In [55]:
# Melhorando a visualização
fig, ax = plt.subplots(figsize=(8,5))

x = np.arange(0,len(valores_grafico))

ax.bar(x,valores_grafico.values,color='tab:gray')

for i in range(0,len(valores_grafico)):
    ax.annotate(round(valores_grafico.values[i],1),(x[i],valores_grafico.values[i]),ha="center",xytext=(0,10),
               textcoords="offset points",c='tab:gray',fontsize=12,fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels([-10,-8,-6,-4,-2,0,2,4,6,8,10])
ax.yaxis.set_visible(False)    
    
ax.spines['top'].set_visible(False)
ax.spines['left'].set_visible(False)
ax.spines['right'].set_visible(False)

ax.annotate('avaliação média',(x[-1],valores_grafico.values[-1]),ha="left",xytext=(20,-15),
               textcoords="offset points",c='tab:gray',fontsize=12,fontweight='bold')
ax.annotate('entrega atrasada <<',(x[0],0),ha="right",xytext=(-20,-15),
               textcoords="offset points",c='tab:gray',fontsize=12,fontweight='bold')
ax.annotate('>> entrega adiantada',(x[-1],0),ha="left",xytext=(+20,-15),
               textcoords="offset points",c='tab:gray',fontsize=12,fontweight='bold')
ax.annotate('entrega no prazo',(x[5],0),ha="center",xytext=(0,-35),
               textcoords="offset points",c='tab:gray',fontsize=12,fontweight='bold')

plt.show()

NameError: name 'plt' is not defined

### <font color='blue'> 3. Como posso saber que as avaliações realmente reclamavam da entrega? </font>

In [ ]:
# Verificando novamente a nossa base já tratada

In [ ]:
# E a base de avaliações

In [ ]:
# Pegando os ids das ordens entre 10 e 5 dias de atraso (as 3 primeiras colunas)

In [ ]:
# Verificando as avaliações desses pedidos

In [ ]:
# Visualizando esses comentários

**Para melhorar a visualização, podemos utilizar uma nuvem de palavras**
- https://pypi.org/project/wordcloud/

In [ ]:
# Instalando o wordcloud 
# !pip install wordcloud

In [ ]:
# Usando a nuvem de palavras

**Para usar a biblioteca, precisamos ter um texto, então podemos transformar todos os comentários em um único texto**

In [ ]:
# Transformando todos os comentários em um único texto

In [ ]:
# Gerando a nuvem de palavras

In [ ]:
# Exibindo a imagem

In [ ]:
# Entendendo melhor o WordCloud

In [ ]:
# E se a gente tentar agrupar por frases?

In [ ]:
# Gerando a nuvem de palavras

In [ ]:
# Exibindo a imagem